# ARMA(p,q) synthetic series vs TimesFM 2.5

**Goal:** Compare one-step-ahead forecasts from a fitted **ARIMA(p,0,q)** model to **TimesFM** on synthetic **ARMA(p,q)** data.

**Setting (same as the AR / MA notebooks):**
- **Horizon:** $t{+}1$ only.
- **Window:** First history length 11 ($y_0,\ldots,y_{10}$), then expanding; $k = 11,\ldots,99$ (**89** test points), $n=100$.
- **Metric:** MSE of one-step errors.
- **DGP:** Gaussian ARMA on the grid **(p,q) ∈ {2,5}²** — four series: **(2,2), (2,5), (5,2), (5,5)**. Classical order matches the DGP: **ARIMA(p,0,q)**.

**Simulation:** we use the usual form  
$y_t = \sum_{i=1}^p \phi_i y_{t-i} + \varepsilon_t + \sum_{j=1}^q \theta_j \varepsilon_{t-j}$  
with i.i.d. standard normal $\varepsilon$, fixed stable coefficients per grid cell, and a short readable loop.

## 1. Setup

**Hugging Face:** `HF_TOKEN` — `pip install -e ".[experiments]"`, `.env` from `.env.example`, or `export HF_TOKEN=...`.

Imports, repo path, `load_dotenv`, fixed **random seed**.

In [6]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import warnings

warnings.filterwarnings("ignore", category=UserWarning, module="statsmodels")

HERE = Path.cwd().resolve()
REPO = HERE if (HERE / "src" / "timesfm").is_dir() else HERE.parent
try:
    from dotenv import load_dotenv
    load_dotenv(REPO / ".env")
except ImportError:
    pass
sys.path.insert(0, str(REPO / "src"))

import timesfm
from statsmodels.tsa.arima.model import ARIMA

RNG_SEED = 42
N = 100
K_FIRST = 11
rng = np.random.default_rng(RNG_SEED)

## 2. Simulate ARMA(p,q) data

For each **(p,q)** we pick fixed **φ** (AR) and **θ** (MA) so the process is reasonably well behaved.  
We draw shocks `e[0], e[1], ...`, seed the first `max(p,q)` values of **y**, then for each later **t** compute **y[t]** from past **y** and past **e** in a straightforward loop.

In [7]:
def coeffs_for(p: int, q: int) -> tuple[np.ndarray, np.ndarray]:
    """Fixed AR(phi) and MA(theta) coefficients for each grid point."""
    if (p, q) == (2, 2):
        phi = np.array([0.25, 0.20])
        theta = np.array([0.35, 0.22])
    elif (p, q) == (2, 5):
        phi = np.array([0.25, 0.20])
        theta = np.array([0.18, 0.14, 0.11, 0.09, 0.07])
    elif (p, q) == (5, 2):
        phi = np.array([0.12, 0.10, 0.08, 0.06, 0.05])
        theta = np.array([0.32, 0.20])
    elif (p, q) == (5, 5):
        phi = np.array([0.12, 0.10, 0.08, 0.06, 0.05])
        theta = np.array([0.16, 0.13, 0.11, 0.09, 0.07])
    else:
        raise ValueError("unsupported (p,q)")
    return phi, theta


def simulate_arma(p: int, q: int, n: int, rng: np.random.Generator) -> np.ndarray:
    phi, theta = coeffs_for(p, q)
    e = rng.normal(0.0, 1.0, n + 20)
    y = np.zeros(n, dtype=np.float64)
    start = max(p, q)
    y[:start] = e[:start]
    for t in range(start, n):
        ar_part = 0.0
        for i in range(1, p + 1):
            ar_part += phi[i - 1] * y[t - i]
        ma_part = e[t]
        for j in range(1, q + 1):
            ma_part += theta[j - 1] * e[t - j]
        y[t] = ar_part + ma_part
    return y


GRID = [(2, 2), (2, 5), (5, 2), (5, 5)]
series: dict[tuple[int, int], np.ndarray] = {}
for p, q in GRID:
    series[(p, q)] = simulate_arma(p, q, N, rng)

## 3. One-step ARMA forecast (classical)

Fit **ARIMA(p,0,q)** on `history` and return the one-step forecast.

**Numerical note:** the default state-space fit can hit **`LinAlgError` (Lyapunov / LU)** when the history is short or the model is stiff (especially **ARMA(5,5)**). We require a **minimum length** (~`p + q + 15`), try **`innovations_mle`** then the default fit, and **skip** that step if both fail (`arma_fit_failures` counts skips). MSE is over **paired** steps where the classical fit succeeded.

In [8]:
def forecast_arma(history: np.ndarray, p: int, q: int) -> float | None:
    h = np.asarray(history, dtype=np.float64).ravel()
    n = len(h)
    # Need enough points so ARMA(p,q) state-space init is well posed
    min_len = p + q + 15
    if n <= max(p, q) or n < min_len:
        return None
    for method in ("innovations_mle", None):
        try:
            mod = ARIMA(h, order=(p, 0, q))
            if method is not None:
                fit = mod.fit(method=method)
            else:
                fit = mod.fit()
            return float(np.asarray(fit.forecast(steps=1)).ravel()[0])
        except (np.linalg.LinAlgError, ValueError, RuntimeError):
            continue
        except Exception:
            continue
    return None

## 4. Load TimesFM (once)

**horizon = 1** in the evaluation loop.

In [9]:
torch.set_float32_matmul_precision("high")
tfm = timesfm.TimesFM_2p5_200M_torch.from_pretrained(
    "google/timesfm-2.5-200m-pytorch",
    torch_compile=False,
)
tfm.compile(
    timesfm.ForecastConfig(
        max_context=1024,
        max_horizon=256,
        normalize_inputs=True,
        use_continuous_quantile_head=True,
        force_flip_invariance=True,
        infer_is_positive=False,
        fix_quantile_crossing=True,
    )
)

## 5. Expanding-window loop and MSE

For each **(p,q)** in the grid, compare **ARIMA(p,0,q)** vs **TimesFM** over $k = 11,\ldots,99$.

In [10]:
rows: list[dict] = []
test_ks = list(range(K_FIRST, N))

for (p, q), y in series.items():
    se_arma: list[float] = []
    se_tf: list[float] = []
    n_fail = 0
    for k in test_ks:
        actual = float(y[k])
        hist = y[:k].astype(np.float32)
        pred_tf = float(tfm.forecast(horizon=1, inputs=[hist])[0][0, 0])
        pred_arma = forecast_arma(y[:k], p, q)
        if pred_arma is None:
            n_fail += 1
            continue
        e_arma = actual - pred_arma
        e_tf = actual - pred_tf
        se_arma.append(e_arma**2)
        se_tf.append(e_tf**2)
    rows.append(
        {
            "p_dgp": p,
            "q_dgp": q,
            "n_test": len(test_ks),
            "n_compared": len(se_arma),
            "arma_fit_failures": n_fail,
            "mse_arma": float(np.mean(se_arma)) if se_arma else float("nan"),
            "mse_timesfm": float(np.mean(se_tf)) if se_tf else float("nan"),
        }
    )

results = pd.DataFrame(rows)
results

/Users/nikhileshbelulkar/Documents/timesfm_google/.venv/lib/python3.11/site-packages/statsmodels/regression/linear_model.py:1717: RuntimeWarning: divide by zero encountered in scalar divide
  return np.dot(wresid, wresid) / self.df_resid
/Users/nikhileshbelulkar/Documents/timesfm_google/.venv/lib/python3.11/site-packages/statsmodels/regression/linear_model.py:1717: RuntimeWarning: divide by zero encountered in scalar divide
  return np.dot(wresid, wresid) / self.df_resid


,p_dgp,q_dgp,n_test,n_compared,arma_fit_failures,mse_arma,mse_timesfm
0,2,2,89,81,8,0.672455,0.693744
1,2,5,89,78,11,0.991613,0.952549
2,5,2,89,78,11,1.292906,1.011070
3,5,5,89,75,14,1.165064,0.960869


## 6. Save table

CSV under `output/`.

In [11]:
out_dir = HERE / "output"
out_dir.mkdir(parents=True, exist_ok=True)
path = out_dir / "arma_vs_timesfm_results.csv"
results.to_csv(path, index=False)
print(path)

/Users/nikhileshbelulkar/Documents/timesfm_google/experiments_nikhilesh_experiments/output/arma_vs_timesfm_results.csv
